<a href="https://colab.research.google.com/github/R4Alex/MIAAD-PADP-Colabfiles/blob/main/Sesion10_Data_Profiling_Entregable_255884.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo:** Alejandro Santillan Arellanes

**Matrícula:** 255884

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [144]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [145]:
print("Columnas: ",df_marketing.columns, "\n")

print("df_marketing Original:\n", df_marketing.head())

Columnas:  Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='object') 

df_marketing Original:
      ID  Year_Birth   Education Marital_Status   Income  Kidhome  Teenhome  \
0  5524        1957  Graduation         Single  58138.0        0         0   
1  2174        1954  Graduation         Single  46344.0        1         1   
2  4141        1965  Graduation       Together  71613.0        0         0   
3  6182        1984  Graduation       Together  26646.0        1         0   
4  5324        1981         PhD        Married  58293

In [146]:
# Tu código aquí
df_marketing_renombrado = df_marketing.copy()

# Correccion estandar a minisculas
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower()

# Renombrado normal para las ultimas columas que no cumplia el estandarizado
df_marketing_renombrado = df_marketing_renombrado.rename(columns={
    'z_costcontact': 'costcontact',
    'z_revenue': 'revenue',
    "dt_customer": "date_customer",
})

df_marketing_renombrado.columns

Index(['id', 'year_birth', 'education', 'marital_status', 'income', 'kidhome',
       'teenhome', 'date_customer', 'recency', 'mntwines', 'mntfruits',
       'mntmeatproducts', 'mntfishproducts', 'mntsweetproducts',
       'mntgoldprods', 'numdealspurchases', 'numwebpurchases',
       'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth',
       'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1',
       'acceptedcmp2', 'complain', 'costcontact', 'revenue', 'response'],
      dtype='object')

---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [147]:
df_netflix.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,14-Aug-20,2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,23-Dec-16,2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,20-Dec-18,2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow..."
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,16-Nov-17,2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi..."
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,01-Jan-20,2008,PG-13,123 min,Dramas,A brilliant group of students become card-coun...


In [148]:
# Tu código aquí

print("Valores nulos Antes: ", df_netflix['date_added'].isnull().sum())
print(df_netflix.dtypes, "\n\n")

df_netflix['date_added'] = pd.to_datetime(df_netflix['date_added'], format='mixed')
print("Valores nulos Despues: ", df_netflix['date_added'].isnull().sum())
print(df_netflix.dtypes)

Valores nulos Antes:  10
show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object 


Valores nulos Despues:  10
show_id                 object
type                    object
title                   object
director                object
cast                    object
country                 object
date_added      datetime64[ns]
release_year             int64
rating                  object
duration                object
listed_in               object
description             object
dtype: object


# Parece que el metod `to_datetime` con format mixed si resolvio correctamente, todos, ya que no se inyectaron valores nulos nuevos

---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [149]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [150]:
# Tu código aquí
df_marketing_dup

print(df_marketing_dup.duplicated().sum())
print(df_marketing_dup.duplicated(subset='ID').sum())

df_marketing_dup = df_marketing_dup.drop_duplicates()

print("Duplicados despues de drop_duplicates:")
print(df_marketing_dup.duplicated().sum())
print(df_marketing_dup.duplicated(subset='ID').sum())

2
2
Duplicados despues de drop_duplicates:
0
0


Aca me quede con la duda, entonces ¿usar `duplicated` sobre todo el dataset, regresa cuantas columas son exactamente iguales? ¿si hay una diferencia no serviria?

---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [151]:
# Tu código aquí

# Cantidad de valores faltantes por columna
print(df_netflix.isnull().sum())

# Filas que tienen al menos un valor faltante
filas_con_nulos = df_netflix.isnull().any(axis=1)

# Cantidad total de filas con al menos un valor faltante
print("Total de filas con valores faltantes:", filas_con_nulos.sum())



show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64
Total de filas con valores faltantes: 2979


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [152]:
# Tu código aquí
# len(df_netflix) da el número total de filas -- lo usamos como base para calcular el porcentaje.
total_filas = len(df_netflix)

# (1 - proporción_de_nulos) * 100 = porcentaje de valores presentes.
completitud = (1 - df_netflix.isnull().sum() / total_filas) * 100
completitud


,0
show_id,100.000000
type,100.000000
title,100.000000
director,69.320663
cast,90.779504
country,93.489149
date_added,99.871581
release_year,100.000000
rating,99.910107
duration,100.000000


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

**En el caso de "Alone" por "Single", creo que los reclasificaría como "Single". En ese caso puntual, es muy evidente la igualdad de la información. En los otros dos casos es más complicado; parecen ser datos "troll" o de broma, por lo que creo que los eliminaría.**

**Como analista de datos, creo que debemos tener criterio y tomar decisiones coherentes dependiendo de cada dataset con el que trabajemos.**

In [153]:
# Tu código aquí

print("Valores por clasificacion originales:")
print(df_marketing["Marital_Status"].value_counts())

df_marketing["Marital_Status"] = df_marketing["Marital_Status"].replace("Alone", "Single")
print("\n\nValores por clasificacion despues:")
print(df_marketing["Marital_Status"].value_counts())


Valores por clasificacion originales:
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64


Valores por clasificacion despues:
Marital_Status
Married     864
Together    580
Single      483
Divorced    232
Widow        77
Absurd        2
YOLO          2
Name: count, dtype: int64


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [154]:
# Tu código aquí

show_id_pattern_ok = df_netflix["show_id"].str.match(r'^s\d+$').sum()

show_id_pattern_ok_percentage = (show_id_pattern_ok / total_filas) * 100

print("total filas:", total_filas)

print("Filas que cumplen:", show_id_pattern_ok)

print("Porcentaje de cumplimiento:", show_id_pattern_ok_percentage)

total filas: 7787
Filas que cumplen: 7787
Porcentaje de cumplimiento: 100.0


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [155]:
# Tu código aquí

print(df_marketing[['Year_Birth']].describe())

df_marketing.sort_values("Year_Birth")[["Year_Birth"]].head(10)

        Year_Birth
count  2240.000000
mean   1968.805804
std      11.984069
min    1893.000000
25%    1959.000000
50%    1970.000000
75%    1977.000000
max    1996.000000


,Year_Birth
239,1893
339,1899
192,1900
1950,1940
424,1941
1923,1943
415,1943
894,1943
39,1943
1150,1943


**Al ordenar los datos, observamos que existen tres registros con valores de `Year_Birth` menores o iguales a 1900. Considero que estos tres valores son errores de captura o, incluso, posibles omisiones en el registro de la información, ya que el año 1900 suele utilizarse como un valor provisional en algunos formularios o sistemas. El siguiente valor observado, 1940, aunque parece relativamente bajo, es mucho más plausible como un año de nacimiento real. Por esta razón, tomaría 1940 como referencia para identificar y eliminar los valores atípicos anteriores.**


---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [156]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [157]:
# Paso 1 — ajuste de tipos
df_practica['Income'] = pd.to_numeric(df_practica['Income'], errors='coerce')
df_practica.dtypes

,0
ID,int64
Year_Birth,int64
Education,object
Marital_Status,object
Income,float64
Kidhome,int64
Teenhome,int64
Dt_Customer,object
Recency,int64
MntWines,int64


In [158]:
# Paso 2 — duplicados
print(df_practica.duplicated().sum())

df_practica = df_practica.drop_duplicates()

print("Duplicados despues de drop_duplicates:")
print(df_practica.duplicated().sum())


1
Duplicados despues de drop_duplicates:
0


In [159]:
# Paso 3 — valores faltantes
print("Valores nulos: ", df_practica.isnull().sum())


Valores nulos:  ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 1
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Z_CostContact          0
Z_Revenue              0
Response               0
dtype: int64


In [160]:
# Paso 4 — exploración categórica
df_practica["Marital_Status"].unique()

array(['Together', 'Single', 'Married', 'Divorced'], dtype=object)

**Tu reporte de profiling:**

*(Durante el profiling se identificó un valor incorrecto en la columna `Income`, por lo que se convirtió la columna a tipo numérico; el valor no válido se transformó en un dato nulo. También se detectó una fila duplicada, la cual fue eliminada para evitar información repetida. Después de estos ajustes, se encontró únicamente un valor nulo correspondiente a `Income`. Finalmente, los valores de `Marital_Status` son consistentes y representan categorías válidas, por lo que considero que no requieren una normalización adicional.
)*